# VisionEdge — Colab GPU Runner (v3)



## 1. Confirm GPU + CUDA version

In [1]:
!nvidia-smi
!nvcc --version

Fri Jul 31 14:21:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload and unzip the project

Select the latest `visionedge.zip` in the file picker.

In [2]:
import glob, os
from google.colab import files

# Remove any previous extraction
!rm -rf /content/visionedge
uploaded = files.upload()
zips = sorted(glob.glob("/content/visionedge*.zip"), key=os.path.getmtime)
latest = zips[-1]
print("Using:", latest)

!unzip -oq "{latest}"
%cd /content/visionedge/backend
!ls

Saving visionedge.zip to visionedge.zip
Using: /content/visionedge.zip
/content/visionedge/backend
benchmark  detector	make_sample_video.py  requirements.txt
core	   __init__.py	orchestration	      streaming
decoder    main.py	pipeline	      tests


## 3. Install dependencies

`tensorrt<11` is pinned deliberately — TensorRT 11 removed the classic
flag-based precision API (`BuilderFlag.FP16`) in favor of strongly-typed
networks, which this project's code does not use. Change `cupy-cuda12x`
if `nvcc --version` above shows a different CUDA major version.

In [3]:
!pip install -q aiohttp aiortc av opencv-python-headless onnx onnxsim ultralytics
!pip install -q "tensorrt<11" pycuda pynvml
!pip install -q cupy-cuda12x
!python -c "import tensorrt as trt; print('TensorRT', trt.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 29.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build de

## 4. Confirm the fixed source files are in place

These two files already have every TensorRT 10.x API fix baked in
(EXPLICIT_BATCH flag removal, platform_has_fast_fp16 removal, ONNX
output-path bug). This cell overwrites them explicitly anyway, so this
notebook is self-contained even against an older zip.

In [4]:
%%writefile detector/export_onnx.py
"""
detector/export_onnx.py

Week 1 (Monday-Tuesday): Model Compilation, stage 1.

Exports a pretrained PyTorch YOLO model (YOLOv10 via Ultralytics) into ONNX
format. This is the CPU-friendly half of "Model Compilation" — no NVIDIA GPU
required to run this file. The output feeds build_engine.py, which DOES
require an NVIDIA GPU + TensorRT.

Usage:
    python -m detector.export_onnx --weights models/yolov10n.pt
"""

import argparse
import logging
import shutil
from pathlib import Path

from core.config import MODEL

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("export_onnx")


def export_to_onnx(weights_path: str, output_path: str, imgsz: tuple, opset: int = 17) -> str:
    """
    Export a PyTorch YOLO checkpoint to ONNX.

    Parameters
    ----------
    weights_path : str
        Path to the .pt file (PyTorch weights).
    output_path : str
        Where to write the .onnx file.
    imgsz : tuple
        (height, width) the network expects.
    opset : int
        ONNX opset version. 17 is a safe default for current TensorRT parsers.

    Returns
    -------
    str
        Path to the exported ONNX file.
    """
    from ultralytics import YOLO  # deferred import: keeps this module importable
                                    # even in environments without ultralytics installed

    log.info("Loading PyTorch weights from %s", weights_path)
    model = YOLO(weights_path)

    log.info("Exporting to ONNX (imgsz=%s, opset=%s)...", imgsz, opset)
    exported_path = model.export(
        format="onnx",
        imgsz=list(imgsz),
        opset=opset,
        dynamic=False,      # static shapes = TensorRT can optimize harder
        simplify=True,      # runs onnx-simplifier to fold constants / clean the graph
    )

    # ultralytics writes the ONNX file next to the source .pt weights by
    # default — it does NOT respect an arbitrary destination path passed
    # here. Move it to where the caller actually asked for it if different.
    exported_path = Path(exported_path)
    output_path = Path(output_path)
    if exported_path.resolve() != output_path.resolve():
        output_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(exported_path), str(output_path))
        log.info("Moved ONNX from %s -> %s", exported_path, output_path)

    log.info("ONNX model written to %s", output_path)
    return str(output_path)


def main():
    parser = argparse.ArgumentParser(description="Export YOLO PyTorch weights to ONNX")
    parser.add_argument("--weights", default=MODEL.pytorch_weights)
    parser.add_argument("--output", default=MODEL.onnx_path)
    parser.add_argument("--imgsz", type=int, nargs=2, default=list(MODEL.input_size))
    parser.add_argument("--opset", type=int, default=17)
    args = parser.parse_args()

    export_to_onnx(args.weights, args.output, tuple(args.imgsz), args.opset)


if __name__ == "__main__":
    main()


Overwriting detector/export_onnx.py


In [5]:
%%writefile detector/build_engine.py
"""
detector/build_engine.py

Week 1 (Thursday): Compile the ONNX graph into a TensorRT engine.

*** REQUIRES AN NVIDIA GPU + TensorRT installed. This will not run on CPU. ***

TensorRT reads the ONNX graph and produces a serialized, hardware-specific
execution plan (an ".engine" file). That plan is tuned for the exact GPU
architecture it was built on — an engine built on a T4 will not load on an
A100; always rebuild the engine on (or for) the target deployment hardware.

Usage:
    python -m detector.build_engine --onnx onnx/yolov10n.onnx --precision fp16
"""

import argparse
import logging

from core.config import MODEL

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("build_engine")


def build_engine(
    onnx_path: str,
    engine_path: str,
    precision: str = "fp16",
    max_workspace_mb: int = 4096,
    calibration_data_dir: str | None = None,
) -> str:
    """
    Compile an ONNX model into a serialized TensorRT engine.

    Parameters
    ----------
    onnx_path : str
        Path to the .onnx file produced by export_onnx.py.
    engine_path : str
        Output path for the serialized .engine file.
    precision : str
        "fp32", "fp16", or "int8". fp16 is the standard tradeoff for
        real-time detection: ~2x throughput vs fp32 with negligible mAP loss.
    max_workspace_mb : int
        Scratch memory TensorRT is allowed to use while searching for the
        fastest kernel implementations during the build (build-time only,
        not needed at inference time).
    calibration_data_dir : str | None
        Directory of representative images, required only if precision="int8".

    Returns
    -------
    str
        Path to the written engine file.
    """
    import tensorrt as trt  # deferred import — this module is NVIDIA-only

    logger = trt.Logger(trt.Logger.INFO)
    builder = trt.Builder(logger)

    # TensorRT <10 required explicitly passing the EXPLICIT_BATCH flag for
    # ONNX-imported networks. TensorRT 10+ kept the NetworkDefinitionCreationFlag
    # class around but removed the EXPLICIT_BATCH member specifically (explicit
    # batch is now the only supported mode, so the flag is meaningless) —
    # checking for the class alone isn't enough, check the member itself.
    has_explicit_batch_flag = (
        hasattr(trt, "NetworkDefinitionCreationFlag")
        and hasattr(trt.NetworkDefinitionCreationFlag, "EXPLICIT_BATCH")
    )
    if has_explicit_batch_flag:
        network_flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        network = builder.create_network(network_flags)
    else:
        network = builder.create_network()  # TensorRT 10+: no flags needed/accepted
    parser = trt.OnnxParser(network, logger)

    log.info("Parsing ONNX file: %s", onnx_path)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                log.error("ONNX parse error: %s", parser.get_error(i))
            raise RuntimeError(f"Failed to parse ONNX model at {onnx_path}")

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, max_workspace_mb * (1 << 20))

    if precision == "fp16":
        # platform_has_fast_fp16 was a purely informational Builder property
        # in TensorRT <10 and was removed in 10+. Setting the FP16 flag
        # below works regardless of whether this check is available, so
        # treat it as optional diagnostics, not a hard requirement.
        if hasattr(builder, "platform_has_fast_fp16") and not builder.platform_has_fast_fp16:
            log.warning("Platform reports no fast FP16 support — build will proceed, "
                        "but throughput gains may be smaller than expected.")
        config.set_flag(trt.BuilderFlag.FP16)

    elif precision == "int8":
        if hasattr(builder, "platform_has_fast_int8") and not builder.platform_has_fast_int8:
            raise RuntimeError("Platform does not support fast INT8.")
        if not calibration_data_dir:
            raise ValueError("INT8 precision requires calibration_data_dir "
                              "(a folder of representative sample frames).")
        config.set_flag(trt.BuilderFlag.INT8)
        config.int8_calibrator = _build_int8_calibrator(calibration_data_dir)

    elif precision != "fp32":
        raise ValueError(f"Unknown precision '{precision}'. Use fp32, fp16, or int8.")

    log.info("Building TensorRT engine (precision=%s). This can take several minutes "
              "the first time — TensorRT is profiling kernel implementations for your "
              "specific GPU.", precision)
    serialized_engine = builder.build_serialized_network(network, config)
    if serialized_engine is None:
        raise RuntimeError("Engine build failed — see TensorRT log output above.")

    with open(engine_path, "wb") as f:
        f.write(serialized_engine)

    log.info("Engine written to %s", engine_path)
    return engine_path


def _build_int8_calibrator(calibration_data_dir: str):
    """
    Minimal INT8 entropy calibrator stub.

    A real implementation loads representative frames, feeds them through
    the network to build an activation histogram, and caches the resulting
    calibration table so future builds are instant. Left as a documented
    stub since Week 1 only targets fp16 per the project spec — wire this up
    if/when you pursue INT8 in a later optimization pass.
    """
    raise NotImplementedError(
        "INT8 calibration is not required for the Week 1 milestone (fp16 is the "
        "target precision). Implement an trt.IInt8EntropyCalibrator2 subclass here "
        "if you extend the project to INT8."
    )


def main():
    parser = argparse.ArgumentParser(description="Build a TensorRT engine from ONNX")
    parser.add_argument("--onnx", default=MODEL.onnx_path)
    parser.add_argument("--engine", default=MODEL.engine_path)
    parser.add_argument("--precision", default=MODEL.precision, choices=["fp32", "fp16", "int8"])
    parser.add_argument("--workspace-mb", type=int, default=4096)
    args = parser.parse_args()

    build_engine(args.onnx, args.engine, args.precision, args.workspace_mb)


if __name__ == "__main__":
    main()


Overwriting detector/build_engine.py


## 5. Get YOLO weights + a sample video

In [6]:
from ultralytics import YOLO
import os

os.makedirs("../models", exist_ok=True)
model = YOLO("yolov10n.pt")   # swap to yolov10s.pt / yolov10x.pt to compare model sizes
!mv yolov10n.pt ../models/yolov10n.pt

os.makedirs("../sample_media", exist_ok=True)
!ffmpeg -y -f lavfi -i testsrc=duration=10:size=1280x720:rate=30 \
    ../sample_media/traffic_4k.mp4 -loglevel error
print("Model and sample video ready.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model and sample video ready.


## 6. Week 1: Export PyTorch -> ONNX

In [7]:
!find . -name "__pycache__" -exec rm -rf {} +
!python -m detector.export_onnx --weights ../models/yolov10n.pt --output ../onnx/yolov10n.onnx

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-07-31 14:56:42,703 [INFO] Loading PyTorch weights from ../models/yolov10n.pt
2026-07-31 14:56:42,921 [INFO] Exporting to ONNX (imgsz=(640, 640), opset=17)...
Ultralytics 8.4.113 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv10n summary (fused): 101 layers, 2,299,264 parameters, 0 gradients, 6.8 GFLOPs

PyTorch: starting from '../models/yolov10n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.6 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not 

## 7. Week 1: Build the TensorRT engine

The real GPU step. First run takes a few minutes.

In [8]:
!find . -name "__pycache__" -exec rm -rf {} +
!python -m detector.build_engine --onnx ../onnx/yolov10n.onnx --engine ../engines/yolov10n_fp16.engine --precision fp16

[07/31/2026-14:56:55] [TRT] [I] [MemUsageChange] Init CUDA: CPU +23, GPU +0, now: CPU 35, GPU 104 (MiB)
2026-07-31 14:56:56,441 [INFO] Parsing ONNX file: ../onnx/yolov10n.onnx
[07/31/2026-14:56:56] [TRT] [I] ----------------------------------------------------------------
[07/31/2026-14:56:56] [TRT] [I] ONNX IR version:  0.0.8
[07/31/2026-14:56:56] [TRT] [I] Opset version:    17
[07/31/2026-14:56:56] [TRT] [I] Producer name:    pytorch
[07/31/2026-14:56:56] [TRT] [I] Producer version: 2.11.0
[07/31/2026-14:56:56] [TRT] [I] Domain:           
[07/31/2026-14:56:56] [TRT] [I] Model version:    0
[07/31/2026-14:56:56] [TRT] [I] Doc string:       
[07/31/2026-14:56:56] [TRT] [I] ----------------------------------------------------------------
2026-07-31 14:56:56,472 [INFO] Building TensorRT engine (precision=fp16). This can take several minutes the first time — TensorRT is profiling kernel implementations for your specific GPU.
[07/31/2026-14:56:56] [TRT] [I] BuilderFlag::kTF32 is set but h

## 8. Mid-Project Review: PyTorch vs TensorRT speedup audit

Screenshot this cell's output for your mentor.

In [9]:
!python -m benchmark.compare --video ../sample_media/traffic_4k.mp4 --weights ../models/yolov10n.pt --engine ../engines/yolov10n_fp16.engine --num-frames 100

2026-07-31 15:00:57,054 [INFO] Loading 100 sample frames from ../sample_media/traffic_4k.mp4
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-07-31 15:01:00,864 [INFO] Loading PyTorch model on device=cuda
2026-07-31 15:01:04,865 [INFO] Loading TensorRT engine from ../engines/yolov10n_fp16.engine
2026-07-31 15:01:04,935 [INFO] Detector ready. Input size=(640, 640)
2026-07-31 15:01:06,803 [INFO] ============================================================
2026-07-31 15:01:06,803 [INFO] PyTorch (native)  fps=80.2  mean=12.47ms  p95=15.92ms
2026-07-31 15:01:06,803 [INFO] TensorRT          fps=228.8  mean=4.37ms  p95=8.02ms
2026-07-31 15:01:06,803 [INFO] ============================================================
2026-07-

## 9. Optional: compare across model sizes

Repeats steps 5-8 for `yolov10s` and `yolov10x` to see how the speedup
changes with model size. Not required — only run if you want the full
three-model comparison table.

In [10]:
for variant in ["yolov10s", "yolov10x"]:
    model = YOLO(f"{variant}.pt")
    !mv {variant}.pt ../models/{variant}.pt
    !python -m detector.export_onnx --weights ../models/{variant}.pt --output ../onnx/{variant}.onnx
    !python -m detector.build_engine --onnx ../onnx/{variant}.onnx --engine ../engines/{variant}_fp16.engine --precision fp16
    !python -m benchmark.compare --video ../sample_media/traffic_4k.mp4 --weights ../models/{variant}.pt --engine ../engines/{variant}_fp16.engine --num-frames 100

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-07-31 15:01:11,308 [INFO] Loading PyTorch weights from ../models/yolov10s.pt
2026-07-31 15:01:11,387 [INFO] Exporting to ONNX (imgsz=(640, 640), opset=17)...
Ultralytics 8.4.113 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv10s summary (fused): 105 layers, 7,248,960 parameters, 0 gradients, 21.7 GFLOPs

PyTorch: starting from '../models/yolov10s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (15.9 MB)

ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.4s, saved as '../models/yolov10s.onnx' (27.9 MB)

Export complete (3.5s)
Results saved to /content/visi

## 10. Save results before the session ends

In [11]:
from google.colab import files

files.download("../engines/yolov10n_fp16.engine")
# files.download("../engines/yolov10s_fp16.engine")
# files.download("../engines/yolov10x_fp16.engine")

print("Also remember: File -> Download -> Download .ipynb to keep this notebook + its printed outputs.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Also remember: File -> Download -> Download .ipynb to keep this notebook + its printed outputs.
